# Lecture 07b: Pandas Methods — View vs Modify, Rename, Drop, Filter, Group

**Dataset:** `predict_heart_disease_train.csv` — 630,000 patients, 14 medical features + target.

### Outline:
1. View method output VS DataFrame modification (assignment needed!).
2. Basic methods: rename columns, handle missing values, drop columns.
3. Replace values in a column using a dictionary.
4. Select data using boolean conditions (AND `&`, OR `|`).
5. Set a column as index.
6. Group by a column or list of columns (`groupby`).

In [1]:
# Present working directory. This is the location of the notebook in your computer.
!pwd

/home/tharg/venv_projects/uoa_py_course/lectures_07_13_pandas_plots_scikit/lecture_07_pandas


In [2]:
import pandas as pd

### Read the heart disease CSV file.
Use `index_col="id"` to avoid the duplicated index issue we saw in Lecture 07a.

In [3]:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
df = pd.read_csv("predict_heart_disease_train.csv", index_col="id")

# Use the code below if you put the data file in the data/ directory.
# df = pd.read_csv("../../data/predict_heart_disease_train.csv", index_col="id")

In [4]:
# 426 rows, 14 columns. All non-null — this dataset is clean (for now).
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      630000 non-null  int64  
 1   Sex                      630000 non-null  int64  
 2   Chest pain type          630000 non-null  int64  
 3   BP                       630000 non-null  int64  
 4   Cholesterol              630000 non-null  int64  
 5   FBS over 120             630000 non-null  int64  
 6   EKG results              630000 non-null  int64  
 7   Max HR                   630000 non-null  int64  
 8   Exercise angina          630000 non-null  int64  
 9   ST depression            630000 non-null  float64
 10  Slope of ST              630000 non-null  int64  
 11  Number of vessels fluro  630000 non-null  int64  
 12  Thallium                 630000 non-null  int64  
 13  Heart Disease            630000 non-null  str    
dtypes: float64(1), 

In [5]:
# Notice: column names have spaces and mixed case — we'll fix that later.
df.head(3)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
id,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence


## 1. View versus "modify" — a critical distinction

Most pandas methods return a **new object** (a "view") and do **NOT** change the original DataFrame.  
To actually modify the DataFrame, you must **assign the result** back to a variable.

This is the single most common source of confusion for beginners.

In [6]:
# Column names — notice the spaces. We'll rename them in Section 2.
df.columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease'],
      dtype='str')

In [7]:
# Selecting a single row returns a Series (not a DataFrame).
# This is patient 0's data.
df.loc[0]

Age                              58
Sex                               1
Chest pain type                   4
BP                              152
Cholesterol                     239
FBS over 120                      0
EKG results                       0
Max HR                          158
Exercise angina                   1
ST depression                   3.6
Slope of ST                       2
Number of vessels fluro           2
Thallium                          7
Heart Disease              Presence
Name: 0, dtype: object

In [8]:
# A single row → pandas Series.
# https://pandas.pydata.org/docs/reference/api/pandas.Series.html
type(df.loc[0])

pandas.Series

In [9]:
# Multiple rows → DataFrame. .loc is inclusive.
df.loc[0:2]

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
id,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence


In [10]:
# Skip the first 5 patients. This returns a VIEW — does NOT modify df.
# The index still starts from 5 (not reset to 0).
df.loc[5:]

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
id,,,,,,,,,,,,,,
5,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
6,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence
7,60,0,3,120,245,0,0,151,0,1.2,1,0,3,Absence
8,48,0,4,140,212,0,2,125,0,0.0,1,0,3,Absence
9,44,0,4,150,197,0,0,150,0,0.0,2,0,3,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,56,0,1,110,226,0,0,132,0,0.0,1,0,7,Absence
629996,54,1,4,128,249,1,2,150,0,0.0,2,0,3,Absence
629997,67,1,4,130,275,0,0,149,0,0.0,1,2,7,Presence


In [11]:
# df is unchanged — the .loc above was just a view!
df.head(2)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
id,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence


In [12]:
# Chaining .reset_index() also does NOT modify df — still just a view.
df.loc[5:].reset_index(drop=True)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence
2,60,0,3,120,245,0,0,151,0,1.2,1,0,3,Absence
3,48,0,4,140,212,0,2,125,0,0.0,1,0,3,Absence
4,44,0,4,150,197,0,0,150,0,0.0,2,0,3,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629990,56,0,1,110,226,0,0,132,0,0.0,1,0,7,Absence
629991,54,1,4,128,249,1,2,150,0,0.0,2,0,3,Absence
629992,67,1,4,130,275,0,0,149,0,0.0,1,2,7,Presence
629993,52,1,4,140,199,0,2,157,0,0.0,1,0,6,Presence


In [13]:
# dStill unchanged! Methods return new objects, they don't modify the original.
df.head(2)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
id,,,,,,,,,,,,,,
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence


### 1.1 To actually modify: create a copy, use assignment, reset the index

Use `.copy()` to create an independent copy. Then use **assignment** (`=`) to store the result.

In [14]:
# .copy() creates an independent copy — changes to copy_df won't affect df.
copy_df = df.copy()

In [15]:
# NOW we modify by assignment: skip first 5 rows, reset index to start from 0.
# drop=True discards the old index (doesn't add it as a column).
copy_df = copy_df.loc[5:].reset_index(drop=True)

In [16]:
# Index now starts from 0. First row is the patient that was previously at index 5.
copy_df.head(2)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence


## 2. Basic methods on a pandas DataFrame

### 2.1 Rename columns

**Why rename?** Our columns have spaces (`"Chest pain type"`) and mixed case (`"Max HR"`).  
This prevents dot notation (`df.Chest pain type` → SyntaxError!) and makes code harder to write.  
**Best practice:** lowercase, underscores, no spaces → `snake_case`.

In [17]:
# View only — rename a single column. Does NOT modify copy_df.
copy_df.rename(columns={"Age": "age"}).head(2)  # rename takes a dict: {"old_name": "new_name"}

,age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence


In [18]:
# Column name is still "Age" — rename() returned a view, not a modification.
copy_df.head(2)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence


In [19]:
# Still the original names — spaces, mixed case.
copy_df.columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease'],
      dtype='str')

In [20]:
# Step 1: Get the current column names as a list.
list_of_column_names = list(copy_df.columns)
list_of_column_names

['Age',
 'Sex',
 'Chest pain type',
 'BP',
 'Cholesterol',
 'FBS over 120',
 'EKG results',
 'Max HR',
 'Exercise angina',
 'ST depression',
 'Slope of ST',
 'Number of vessels fluro',
 'Thallium',
 'Heart Disease']

In [21]:
# # Dict comprehension — creates the skeleton, but you'd still need to fill in new names.
# dict_of_column_renames = {col: "new_name" for col in list_of_column_names}
# dict_of_column_renames

In [22]:
# Step 2: Create a list of new snake_case names.
# Rule: lowercase, underscores instead of spaces, no special characters.
list_of_new_column_names = [
    "age", "sex", "chest_pain_type", "bp", "cholesterol",
    "fbs_over_120", "ekg_results", "max_hr", "exercise_angina",
    "st_depression", "slope_of_st", "num_vessels_fluro", "thallium", "heart_disease",
]

list_of_new_column_names

['age',
 'sex',
 'chest_pain_type',
 'bp',
 'cholesterol',
 'fbs_over_120',
 'ekg_results',
 'max_hr',
 'exercise_angina',
 'st_depression',
 'slope_of_st',
 'num_vessels_fluro',
 'thallium',
 'heart_disease']

In [23]:
# Step 3: zip the two lists and create a dictionary: {"old_name": "new_name"}.
dict_of_column_renames = dict(
    zip(list_of_column_names, list_of_new_column_names)
)

In [24]:
# The mapping: original name → snake_case name.
dict_of_column_renames

{'Age': 'age',
 'Sex': 'sex',
 'Chest pain type': 'chest_pain_type',
 'BP': 'bp',
 'Cholesterol': 'cholesterol',
 'FBS over 120': 'fbs_over_120',
 'EKG results': 'ekg_results',
 'Max HR': 'max_hr',
 'Exercise angina': 'exercise_angina',
 'ST depression': 'st_depression',
 'Slope of ST': 'slope_of_st',
 'Number of vessels fluro': 'num_vessels_fluro',
 'Thallium': 'thallium',
 'Heart Disease': 'heart_disease'}

In [25]:
# Preview: rename returns a VIEW. copy_df is NOT modified yet.
copy_df.rename(columns=dict_of_column_renames).head(2)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence


In [26]:
# Confirm: columns are still the original names.
copy_df.columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease'],
      dtype='str')

In [27]:
# NOW we modify by assignment. This actually changes copy_df.
# Warning: don't chain .head(2) here or you'll overwrite copy_df with just 2 rows!
copy_df = copy_df.rename(columns=dict_of_column_renames)

In [28]:
# copy_df.rename?

In [29]:
# Now columns are snake_case. Dot notation works: copy_df.age, copy_df.bp, etc.
copy_df.head(3)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence
2,60,0,3,120,245,0,0,151,0,1.2,1,0,3,Absence


In [30]:
copy_df.max_hr.mean()

np.float64(152.8167779109358)

### 2.2 Find and handle missing values (NaN)

In real-world data, missing values are **very common**. Pandas represents them as `NaN` (Not a Number).

Our heart disease dataset is clean (no NaN), so we'll first **verify** that, then **intentionally inject** some NaN values to practice the `dropna()` workflow.

Advanced Reading: [Is inplace harmful or not?](https://stackoverflow.com/a/59242208)

In [31]:
# .isnull() returns True/False for every cell. All False → no missing data.
copy_df.isnull()

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629990,False,False,False,False,False,False,False,False,False,False,False,False,False,False
629991,False,False,False,False,False,False,False,False,False,False,False,False,False,False
629992,False,False,False,False,False,False,False,False,False,False,False,False,False,False
629993,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [32]:
# .any(axis=0): is there at least one NaN per COLUMN? All False → clean data.
copy_df.isnull().any(axis=0)

age                  False
sex                  False
chest_pain_type      False
bp                   False
cholesterol          False
fbs_over_120         False
ekg_results          False
max_hr               False
exercise_angina      False
st_depression        False
slope_of_st          False
num_vessels_fluro    False
thallium             False
heart_disease        False
dtype: bool

In [33]:
# .any(axis=1): is there at least one NaN per ROW? All False → clean data.
copy_df.isnull().any(axis=1)

0         False
1         False
2         False
3         False
4         False
          ...  
629990    False
629991    False
629992    False
629993    False
629994    False
Length: 629995, dtype: bool

In [34]:
# Filter rows that have ANY NaN — empty DataFrame because our data is clean.
copy_df[copy_df.isnull().any(axis=1)]

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease


In [35]:
# Let's intentionally inject NaN to practice the dropna workflow.
import numpy as np

# Create a dirty copy. Set some cholesterol values to NaN.
dirty_df = copy_df.copy()

dirty_df.loc[0:2, "cholesterol"] = np.nan    # rows 0, 1, 2
dirty_df.loc[10:12, "bp"] = np.nan           # rows 10, 11, 12
dirty_df.head(13)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,1,4,138.0,NaN,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130.0,NaN,0,2,152,0,0.8,2,2,3,Presence
2,60,0,3,120.0,NaN,0,0,151,0,1.2,1,0,3,Absence
3,48,0,4,140.0,212.0,0,2,125,0,0.0,1,0,3,Absence
4,44,0,4,150.0,197.0,0,0,150,0,0.0,2,0,3,Absence
5,41,1,2,120.0,212.0,0,0,173,0,0.6,2,0,3,Absence
6,52,0,2,140.0,234.0,0,0,160,0,0.0,1,0,3,Absence
7,41,0,3,108.0,230.0,0,2,142,0,0.6,2,1,3,Absence
8,44,1,3,110.0,263.0,0,2,161,0,0.0,1,0,3,Absence
9,42,1,4,108.0,244.0,0,0,150,1,1.6,2,0,7,Presence


In [36]:
# Now isnull().any() shows which columns have missing values.
dirty_df.isnull().any(axis=0)

age                  False
sex                  False
chest_pain_type      False
bp                    True
cholesterol           True
fbs_over_120         False
ekg_results          False
max_hr               False
exercise_angina      False
st_depression        False
slope_of_st          False
num_vessels_fluro    False
thallium             False
heart_disease        False
dtype: bool

In [37]:
type(dirty_df[dirty_df.isnull().any(axis=1)])

pandas.DataFrame

In [38]:
# Show only rows with NaN. 6 rows have missing values.
dirty_df[dirty_df.isnull().any(axis=1)]

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,1,4,138.0,NaN,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,130.0,NaN,0,2,152,0,0.8,2,2,3,Presence
2,60,0,3,120.0,NaN,0,0,151,0,1.2,1,0,3,Absence
10,53,1,1,NaN,231.0,0,0,160,0,0.2,2,0,7,Absence
11,50,1,2,NaN,274.0,0,2,163,0,1.8,2,0,3,Absence
12,56,1,4,NaN,282.0,0,2,162,1,1.6,2,2,3,Presence


In [39]:
# .info() shows the non-null counts — cholesterol and bp now have fewer than 421.
dirty_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 629995 entries, 0 to 629994
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   age                629995 non-null  int64  
 1   sex                629995 non-null  int64  
 2   chest_pain_type    629995 non-null  int64  
 3   bp                 629992 non-null  float64
 4   cholesterol        629992 non-null  float64
 5   fbs_over_120       629995 non-null  int64  
 6   ekg_results        629995 non-null  int64  
 7   max_hr             629995 non-null  int64  
 8   exercise_angina    629995 non-null  int64  
 9   st_depression      629995 non-null  float64
 10  slope_of_st        629995 non-null  int64  
 11  num_vessels_fluro  629995 non-null  int64  
 12  thallium           629995 non-null  int64  
 13  heart_disease      629995 non-null  str    
dtypes: float64(3), int64(10), str(1)
memory usage: 67.3 MB


In [40]:
# Uncomment to see dropna() documentation:
# dirty_df.dropna?

In [41]:
# dropna(axis=0): drop ROWS with any NaN. Just a view!
dirty_df.dropna(axis=1)

,age,sex,chest_pain_type,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,1,4,0,0,147,1,1.6,2,2,7,Presence
1,59,1,4,0,2,152,0,0.8,2,2,3,Presence
2,60,0,3,0,0,151,0,1.2,1,0,3,Absence
3,48,0,4,0,2,125,0,0.0,1,0,3,Absence
4,44,0,4,0,0,150,0,0.0,2,0,3,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...
629990,56,0,1,0,0,132,0,0.0,1,0,7,Absence
629991,54,1,4,1,2,150,0,0.0,2,0,3,Absence
629992,67,1,4,0,0,149,0,0.0,1,2,7,Presence
629993,52,1,4,0,2,157,0,0.0,1,0,6,Presence


In [42]:
# dropna(axis=1): drop COLUMNS that contain any NaN. Removes cholesterol and bp entirely!
dirty_df.dropna(axis=0)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
3,48,0,4,140.0,212.0,0,2,125,0,0.0,1,0,3,Absence
4,44,0,4,150.0,197.0,0,0,150,0,0.0,2,0,3,Absence
5,41,1,2,120.0,212.0,0,0,173,0,0.6,2,0,3,Absence
6,52,0,2,140.0,234.0,0,0,160,0,0.0,1,0,3,Absence
7,41,0,3,108.0,230.0,0,2,142,0,0.6,2,1,3,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629990,56,0,1,110.0,226.0,0,0,132,0,0.0,1,0,7,Absence
629991,54,1,4,128.0,249.0,1,2,150,0,0.0,2,0,3,Absence
629992,67,1,4,130.0,275.0,0,0,149,0,0.0,1,2,7,Presence
629993,52,1,4,140.0,199.0,0,2,157,0,0.0,1,0,6,Presence


In [43]:
# dirty_df is STILL unchanged — dropna() returned views, not modifications.
len(dirty_df)

629995

In [44]:
# Avoid inplace=True — it's discouraged and may be deprecated.
# dirty_df.dropna(inplace=True)  # DON'T do this

In [45]:
# Modify by assignment: drop rows with NaN.
dirty_df = dirty_df.dropna()

In [46]:
# 6 rows dropped (3 with NaN cholesterol + 3 with NaN bp).
len(dirty_df)

629989

In [47]:
# The index is NOT reset — gaps at rows 0, 1, 2, 10, 11, 12.
# Always mind the index after dropping rows!
dirty_df.head(5)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
3,48,0,4,140.0,212.0,0,2,125,0,0.0,1,0,3,Absence
4,44,0,4,150.0,197.0,0,0,150,0,0.0,2,0,3,Absence
5,41,1,2,120.0,212.0,0,0,173,0,0.6,2,0,3,Absence
6,52,0,2,140.0,234.0,0,0,160,0,0.0,1,0,3,Absence
7,41,0,3,108.0,230.0,0,2,142,0,0.6,2,1,3,Absence


In [48]:
# reset_index() WITHOUT drop=True: the old index becomes a new column!
dirty_df = dirty_df.reset_index()  # try: .reset_index(drop=True) to discard the old index.

In [49]:
# A new "index" column appeared — the old index values. Usually unwanted.
dirty_df.head(20)

,index,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,3,48,0,4,140.0,212.0,0,2,125,0,0.0,1,0,3,Absence
1,4,44,0,4,150.0,197.0,0,0,150,0,0.0,2,0,3,Absence
2,5,41,1,2,120.0,212.0,0,0,173,0,0.6,2,0,3,Absence
3,6,52,0,2,140.0,234.0,0,0,160,0,0.0,1,0,3,Absence
4,7,41,0,3,108.0,230.0,0,2,142,0,0.6,2,1,3,Absence
5,8,44,1,3,110.0,263.0,0,2,161,0,0.0,1,0,3,Absence
6,9,42,1,4,108.0,244.0,0,0,150,1,1.6,2,0,7,Presence
7,13,65,0,4,140.0,197.0,0,0,161,0,2.2,2,1,3,Presence
8,14,46,0,3,178.0,199.0,0,0,169,0,0.0,1,0,3,Absence
9,15,62,1,4,110.0,197.0,0,0,172,0,2.2,2,3,3,Presence


In [50]:
# Notice the "index" column at the left — old index values retained as data.
dirty_df.tail(5)

,index,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
629984,629990,56,0,1,110.0,226.0,0,0,132,0,0.0,1,0,7,Absence
629985,629991,54,1,4,128.0,249.0,1,2,150,0,0.0,2,0,3,Absence
629986,629992,67,1,4,130.0,275.0,0,0,149,0,0.0,1,2,7,Presence
629987,629993,52,1,4,140.0,199.0,0,2,157,0,0.0,1,0,6,Presence
629988,629994,51,0,2,130.0,199.0,0,0,168,0,0.0,1,0,3,Absence


### 2.3 Drop a column by name

Use `.drop("column_name", axis=1)` to remove a column. `axis=1` means "operate on columns".

In [51]:
# Drop the unwanted "index" column from dirty_df (created by reset_index).
dirty_df = dirty_df.drop("index", axis=1)

In [52]:
# "index" column is gone. We continue with copy_df (the clean version) for the rest.
dirty_df.tail(2)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
629987,52,1,4,140.0,199.0,0,2,157,0,0.0,1,0,6,Presence
629988,51,0,2,130.0,199.0,0,0,168,0,0.0,1,0,3,Absence


## 3. Replace values in a column, using a dictionary

Encoded values like `Sex=0/1` are common in medical datasets but not human-readable.  
We use `.replace()` with a dictionary to map coded values → meaningful labels.

[map or replace with a dict — StackOverflow](https://stackoverflow.com/a/49259581)

In [53]:
copy_df.age.unique()

array([38, 59, 60, 48, 44, 41, 52, 42, 53, 50, 56, 65, 46, 62, 57, 54, 66,
       51, 55, 43, 71, 63, 61, 35, 58, 49, 47, 67, 64, 45, 40, 70, 69, 37,
       76, 34, 68, 39, 74, 77, 29, 75])

In [54]:
# What unique values does the "sex" column have?
copy_df.sex.unique()  # 0 and 1 — not very informative!

array([1, 0])

In [55]:
# Inline approach — works but harder to read with many replacements:
# copy_df.sex = copy_df.sex.replace({1: "Male", 0: "Female"})

In [56]:
# Better: define the mapping as a separate dictionary.
sex_replacements = {1: "Male", 0: "Female"}

# Also prepare replacements for other encoded columns.
fbs_replacements = {1: "Yes", 0: "No"}
angina_replacements = {1: "Yes", 0: "No"}

In [57]:
# Apply the replacements. This MODIFIES copy_df (assignment to column).
copy_df.sex = copy_df.sex.replace(sex_replacements)

copy_df.fbs_over_120 = copy_df.fbs_over_120.replace(fbs_replacements)

copy_df.exercise_angina = copy_df.exercise_angina.replace(angina_replacements)

In [58]:
# Verify: sex now shows "Male" and "Female" instead of 1 and 0.
copy_df.sex.unique()

array(['Male', 'Female'], dtype=object)

In [59]:
# Much more readable! Sex, fbs_over_120, exercise_angina are now descriptive.
copy_df.sample(10)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
350352,53,Male,2,110,245,Yes,0,170,No,0.6,1,0,3,Absence
25340,63,Female,4,120,204,No,2,152,No,0.0,1,0,7,Presence
283790,56,Male,4,120,245,No,2,179,No,0.0,2,1,7,Presence
101779,41,Male,4,130,244,No,2,111,Yes,3.2,2,0,7,Presence
32814,61,Female,3,136,256,No,0,161,No,0.9,1,1,3,Absence
262738,60,Male,4,130,286,No,0,160,Yes,0.0,1,0,3,Presence
223602,57,Male,4,145,234,No,2,158,No,2.2,2,3,7,Presence
444466,54,Male,4,110,256,No,0,125,Yes,0.0,2,1,3,Presence
542778,49,Female,2,126,283,No,2,162,No,2.4,1,0,3,Absence
242385,58,Male,4,120,234,No,2,142,Yes,1.6,2,1,7,Presence


## 4. Select data using boolean conditions

Boolean indexing: create a True/False mask, then use it to filter rows.  
This is one of the most powerful and frequently used pandas operations.

In [60]:
# Step 1: Create a boolean mask — True where condition is met, False otherwise.
copy_df["heart_disease"] == "Presence"

0          True
1          True
2         False
3         False
4         False
          ...  
629990    False
629991    False
629992     True
629993     True
629994    False
Name: heart_disease, Length: 629995, dtype: bool

In [61]:
# Step 2: Use the mask to filter — only patients WITH heart disease.
copy_df[copy_df["heart_disease"] == "Presence"]

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7,Presence
1,59,Male,4,130,246,No,2,152,No,0.8,2,2,3,Presence
9,42,Male,4,108,244,No,0,150,Yes,1.6,2,0,7,Presence
12,56,Male,4,160,282,No,2,162,Yes,1.6,2,2,3,Presence
13,65,Female,4,140,197,No,0,161,No,2.2,2,1,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629986,59,Male,3,140,263,Yes,2,160,No,2.0,2,2,7,Presence
629987,48,Male,1,140,229,No,2,142,No,1.2,1,0,7,Presence
629989,58,Male,4,150,235,No,0,162,No,0.2,1,2,3,Presence
629992,67,Male,4,130,275,No,0,149,No,0.0,1,2,7,Presence


In [62]:
# How many patients have heart disease?
len(copy_df[copy_df["heart_disease"] == "Presence"])

282452

In [63]:
# Same result with dot notation (works because "heart_disease" has no spaces).
copy_df[copy_df.heart_disease == "Presence"]

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7,Presence
1,59,Male,4,130,246,No,2,152,No,0.8,2,2,3,Presence
9,42,Male,4,108,244,No,0,150,Yes,1.6,2,0,7,Presence
12,56,Male,4,160,282,No,2,162,Yes,1.6,2,2,3,Presence
13,65,Female,4,140,197,No,0,161,No,2.2,2,1,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629986,59,Male,3,140,263,Yes,2,160,No,2.0,2,2,7,Presence
629987,48,Male,1,140,229,No,2,142,No,1.2,1,0,7,Presence
629989,58,Male,4,150,235,No,0,162,No,0.2,1,2,3,Presence
629992,67,Male,4,130,275,No,0,149,No,0.0,1,2,7,Presence


In [64]:
# Numeric condition: patients older than 60.
copy_df[copy_df.age > 60]

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
13,65,Female,4,140,197,No,0,161,No,2.2,2,1,3,Presence
15,62,Male,4,110,197,No,0,172,No,2.2,2,3,3,Presence
16,65,Male,3,120,226,No,0,145,Yes,0.0,2,0,3,Presence
22,66,Male,4,110,234,No,2,122,No,0.0,1,0,7,Absence
33,71,Male,4,100,229,No,2,154,No,0.0,1,1,7,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629970,65,Male,4,140,274,No,0,132,Yes,1.0,2,0,7,Presence
629971,62,Female,3,125,270,No,2,147,Yes,0.0,1,0,3,Absence
629982,62,Male,4,180,234,No,2,128,Yes,1.2,2,1,7,Presence
629984,61,Male,3,150,234,No,2,161,Yes,1.0,2,0,3,Presence


In [65]:
# Store filtered results. Notice: the index is NOT reset (gaps in row numbers).
df_sick = copy_df[copy_df["heart_disease"] == "Presence"]

In [66]:
df_sick

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7,Presence
1,59,Male,4,130,246,No,2,152,No,0.8,2,2,3,Presence
9,42,Male,4,108,244,No,0,150,Yes,1.6,2,0,7,Presence
12,56,Male,4,160,282,No,2,162,Yes,1.6,2,2,3,Presence
13,65,Female,4,140,197,No,0,161,No,2.2,2,1,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629986,59,Male,3,140,263,Yes,2,160,No,2.0,2,2,7,Presence
629987,48,Male,1,140,229,No,2,142,No,1.2,1,0,7,Presence
629989,58,Male,4,150,235,No,0,162,No,0.2,1,2,3,Presence
629992,67,Male,4,130,275,No,0,149,No,0.0,1,2,7,Presence


### 4.1 The OR operator: `|`

Use OR to create a **union** — patients matching **any** of the conditions.

In [67]:
(copy_df[
    (copy_df.age == 64)
    # | (copy_df.age == 44)df_male_sick
    | (copy_df.age == 54)
    | (copy_df.age == 34)
])

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
21,54,Male,3,120,250,No,0,163,No,0.3,1,0,3,Absence
43,54,Male,3,130,226,No,2,145,Yes,0.9,1,0,3,Absence
50,54,Male,4,130,295,No,2,139,No,1.8,2,1,7,Presence
63,54,Female,3,108,249,No,0,178,No,0.0,1,0,3,Absence
65,64,Female,3,120,211,No,0,162,No,0.0,1,0,7,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629940,54,Male,3,150,219,No,0,162,No,1.5,1,0,3,Absence
629953,54,Male,2,138,226,No,2,96,Yes,3.4,1,1,3,Absence
629957,54,Male,2,120,250,No,0,170,Yes,2.6,2,0,7,Presence
629960,54,Male,3,110,229,No,0,146,No,0.0,1,0,3,Absence


In [68]:
# Patients with chest pain type 3 OR 4 (the more severe types).
# Each condition must be in parentheses when using | or &.
(copy_df[
    (copy_df.chest_pain_type == 3)
    # | (copy_df.chest_pain_type == 4)
])

# OR gives the UNION: rows matching EITHER condition.

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
2,60,Female,3,120,245,No,0,151,No,1.2,1,0,3,Absence
7,41,Female,3,108,230,No,2,142,No,0.6,2,1,3,Absence
8,44,Male,3,110,263,No,2,161,No,0.0,1,0,3,Absence
14,46,Female,3,178,199,No,0,169,No,0.0,1,0,3,Absence
16,65,Male,3,120,226,No,0,145,Yes,0.0,2,0,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629978,41,Male,3,140,243,No,2,181,No,0.0,2,2,3,Absence
629981,59,Female,3,180,258,No,0,153,No,0.0,1,0,3,Absence
629984,61,Male,3,150,234,No,2,161,Yes,1.0,2,0,3,Presence
629986,59,Male,3,140,263,Yes,2,160,No,2.0,2,2,7,Presence


### 4.2 The AND operator: `&`

Use AND to create an **intersection** — patients matching **all** conditions simultaneously.

In [69]:
# Male patients with heart disease.
copy_df[
    (copy_df.sex == "Male")
    & (copy_df.heart_disease == "Presence")
]

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7,Presence
1,59,Male,4,130,246,No,2,152,No,0.8,2,2,3,Presence
9,42,Male,4,108,244,No,0,150,Yes,1.6,2,0,7,Presence
12,56,Male,4,160,282,No,2,162,Yes,1.6,2,2,3,Presence
15,62,Male,4,110,197,No,0,172,No,2.2,2,3,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629986,59,Male,3,140,263,Yes,2,160,No,2.0,2,2,7,Presence
629987,48,Male,1,140,229,No,2,142,No,1.2,1,0,7,Presence
629989,58,Male,4,150,235,No,0,162,No,0.2,1,2,3,Presence
629992,67,Male,4,130,275,No,0,149,No,0.0,1,2,7,Presence


In [70]:
# Union: patients older than 65 OR with high cholesterol (> 300).
copy_df[
    (copy_df.age > 65)
    | (copy_df.cholesterol > 300)
]

# Union = "either or both" conditions are True.

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
22,66,Male,4,110,234,No,2,122,No,0.0,1,0,7,Absence
26,53,Male,4,140,303,No,0,168,No,0.6,2,2,7,Presence
33,71,Male,4,100,229,No,2,154,No,0.0,1,1,7,Presence
46,50,Male,4,128,303,No,2,166,No,3.0,3,1,7,Presence
54,49,Female,4,120,304,Yes,2,150,No,1.0,2,0,7,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629889,41,Female,3,140,309,No,0,140,No,0.8,1,0,3,Absence
629914,59,Male,3,130,326,No,2,165,No,0.2,1,0,3,Absence
629943,67,Female,3,132,201,No,2,160,No,0.0,1,0,3,Absence
629949,44,Male,4,120,305,No,0,147,No,0.0,1,0,3,Absence


In [71]:
# Intersection: patients older than 55 AND with high blood pressure (> 140).
copy_df[
    (copy_df.age > 55)
    & (copy_df.bp > 140)
]

# Intersection = BOTH conditions must be True.

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
12,56,Male,4,160,282,No,2,162,Yes,1.6,2,2,3,Presence
37,60,Male,4,172,204,No,0,159,No,0.0,1,0,3,Absence
38,61,Female,4,160,249,No,0,159,No,0.0,1,0,3,Absence
64,63,Male,4,180,228,No,2,131,Yes,3.2,2,0,7,Presence
68,64,Male,3,145,244,No,0,178,Yes,0.0,1,1,7,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629963,57,Male,3,160,250,No,0,151,No,0.0,1,0,3,Absence
629981,59,Female,3,180,258,No,0,153,No,0.0,1,0,3,Absence
629982,62,Male,4,180,234,No,2,128,Yes,1.2,2,1,7,Presence
629984,61,Male,3,150,234,No,2,161,Yes,1.0,2,0,3,Presence


In [72]:
# Store filtered result and reset the index.
df_male_sick = copy_df[
    (copy_df.sex == "Male")
    & (copy_df.heart_disease == "Presence")
].reset_index(drop=True)

df_male_sick

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7,Presence
1,59,Male,4,130,246,No,2,152,No,0.8,2,2,3,Presence
2,42,Male,4,108,244,No,0,150,Yes,1.6,2,0,7,Presence
3,56,Male,4,160,282,No,2,162,Yes,1.6,2,2,3,Presence
4,62,Male,4,110,197,No,0,172,No,2.2,2,3,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250320,59,Male,3,140,263,Yes,2,160,No,2.0,2,2,7,Presence
250321,48,Male,1,140,229,No,2,142,No,1.2,1,0,7,Presence
250322,58,Male,4,150,235,No,0,162,No,0.2,1,2,3,Presence
250323,67,Male,4,130,275,No,0,149,No,0.0,1,2,7,Presence


## 5. Set a column as index

In [84]:
# Set "heart_disease" as the index (replaces the default integer index).
# This is a VIEW — does NOT modify copy_df.
copy_df.set_index("heart_disease")

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium
heart_disease,,,,,,,,,,,,,
Presence,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7
Presence,59,Male,4,130,246,No,2,152,No,0.8,2,2,3
Absence,60,Female,3,120,245,No,0,151,No,1.2,1,0,3
Absence,48,Female,4,140,212,No,2,125,No,0.0,1,0,3
Absence,44,Female,4,150,197,No,0,150,No,0.0,2,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Absence,56,Female,1,110,226,No,0,132,No,0.0,1,0,7
Absence,54,Male,4,128,249,Yes,2,150,No,0.0,2,0,3
Presence,67,Male,4,130,275,No,0,149,No,0.0,1,2,7


In [74]:
# copy_df is unchanged — set_index returned a view, not a modification.
copy_df

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,num_vessels_fluro,thallium,heart_disease
0,38,Male,4,138,283,No,0,147,Yes,1.6,2,2,7,Presence
1,59,Male,4,130,246,No,2,152,No,0.8,2,2,3,Presence
2,60,Female,3,120,245,No,0,151,No,1.2,1,0,3,Absence
3,48,Female,4,140,212,No,2,125,No,0.0,1,0,3,Absence
4,44,Female,4,150,197,No,0,150,No,0.0,2,0,3,Absence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629990,56,Female,1,110,226,No,0,132,No,0.0,1,0,7,Absence
629991,54,Male,4,128,249,Yes,2,150,No,0.0,2,0,3,Absence
629992,67,Male,4,130,275,No,0,149,No,0.0,1,2,7,Presence
629993,52,Male,4,140,199,No,2,157,No,0.0,1,0,6,Presence


## 6. Group by a column or a list of columns

`groupby()` splits the DataFrame into groups, then applies an aggregation (e.g., `.size()`, `.mean()`).

In [87]:
# Group by heart_disease → count patients in each category.
# .size() returns a Series. Suited for categorical columns.
by_disease = copy_df.groupby(
    by="heart_disease"
    ).size().sort_values(ascending=False)

by_disease

heart_disease
Absence     347543
Presence    282452
dtype: int64

In [88]:
# Group by heart_disease → count patients in each category.
# .size() returns a Series. Suited for categorical columns.
by_disease = copy_df.groupby(by="age").size()#.sort_values(ascending=False)

by_disease

age
29     1048
34     2623
35     3991
37     2481
38      931
39     5540
40     4554
41    24633
42    21958
43    16231
44    25475
45    14629
46    16246
47     6888
48    16282
49    10643
50    16659
51    37526
52    30571
53    15663
54    46826
55    11424
56    21148
57    32442
58    41712
59    31713
60    29437
61    13826
62    25872
63    13184
64    20109
65    17454
66    11771
67    17832
68     3849
69     4018
70     5950
71     4556
74      741
75        1
76      798
77      760
dtype: int64

In [89]:
# Same result, more verbose — useful for debugging step by step.
by_disease = copy_df.groupby(by="heart_disease").size()
by_disease = by_disease.sort_values(ascending=False)

by_disease

heart_disease
Absence     347543
Presence    282452
dtype: int64

In [90]:
# .head(n) works on Series too.
by_disease.head(2)

heart_disease
Absence     347543
Presence    282452
dtype: int64

In [91]:
# groupby().size() returns a pandas Series, not a DataFrame.
type(by_disease)

pandas.Series

In [94]:
# Group by sex — how many Male vs Female patients?
by_sex = copy_df.groupby("sex").size().sort_values()

by_sex

sex
Female    179715
Male      450280
dtype: int64

In [93]:
# Group by TWO columns: sex first, then heart_disease.
# The order of columns matters — it determines the hierarchy.
by_sex_disease = copy_df.groupby(
    ["sex", "heart_disease"]
    ).size().sort_values(ascending=False)

by_sex_disease

sex     heart_disease
Male    Presence         250325
        Absence          199955
Female  Absence          147588
        Presence          32127
dtype: int64

In [ ]:
# Reverse the column order: heart_disease first, then sex.
by_disease_sex = copy_df.groupby(
    by=["heart_disease", "sex"]
    ).size().sort_values(ascending=False)

by_disease_sex

heart_disease  sex   
Presence       Male      250325
Absence        Male      199955
               Female    147588
Presence       Female     32127
dtype: int64

In [96]:
# Convert the Series to a DataFrame for easier manipulation.
by_sex_disease = by_sex_disease.to_frame()
by_sex_disease

0
sex    heart_disease        
Male   Presence       250325
       Absence        199955
Female Absence        147588
       Presence        32127

In [83]:
# Now it's a DataFrame, not a Series.
type(by_sex_disease)

pandas.DataFrame

#### Resources

- [pandas documentation — DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [pandas documentation — groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [pandas documentation — replace](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html)
- [Boolean indexing — pandas user guide](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)

In [97]:
# Save a result to a CSV file. By default, the index is included as a column in the output.
by_sex_disease.to_csv("by_sex_disease.csv")

In [98]:
# save copy_df to a CSV file. By default, the index is included as a column in the output.
copy_df.to_csv("copy_df.csv")